In [ ]:
# Example 1: @beartype decorator on a simple typed function — happy path.
from beartype import beartype

@beartype
def greet(name: str, times: int) -> str:
    return (name + ' ') * times

result = greet('hello', 3)
print(repr(result))
assert result == 'hello hello hello '
assert isinstance(result, str)

In [ ]:
# Example 2: @beartype raises on a bad-type argument.
from beartype import beartype
from beartype.roar import BeartypeCallHintParamViolation

@beartype
def add(x: int, y: int) -> int:
    return x + y

assert add(2, 3) == 5

raised = False
try:
    add('two', 3)  # type: ignore[arg-type]
except BeartypeCallHintParamViolation as e:
    raised = True
    print(f"Got expected violation: {type(e).__name__}")

assert raised, 'beartype should have rejected the str argument'

In [ ]:
# Example 3: complex nested generic type hints (list[dict[str, int]]).
from beartype import beartype
from beartype.roar import BeartypeCallHintParamViolation

@beartype
def total_scores(rows: list[dict[str, int]]) -> int:
    return sum(v for row in rows for v in row.values())

good = [{'a': 1, 'b': 2}, {'c': 3}]
assert total_scores(good) == 6

raised = False
try:
    total_scores([{'a': 'not-an-int'}])  # type: ignore[list-item]
except BeartypeCallHintParamViolation:
    raised = True
assert raised

In [ ]:
# Example 4: door API (procedural runtime type-checks: is_bearable, die_if_unbearable).
from beartype.door import is_bearable, die_if_unbearable
from beartype.roar import BeartypeDoorHintViolation

# NOTE: is_bearable defaults to the O(1) strategy, which type-checks only a random
# *sample* of a container's items — so `is_bearable([1, 'two', 3], list[int])` is
# non-deterministic (may miss the bad element). Use deterministic mismatches: a
# wrong container type and a wrong scalar type are always rejected.
assert is_bearable([1, 2, 3], list[int]) is True
assert is_bearable('hello', list[int]) is False   # str is not a list — deterministic
assert is_bearable(123, str) is False              # int is not a str — deterministic
assert is_bearable({'k': 1}, dict[str, int]) is True

die_if_unbearable([1, 2, 3], list[int])  # should NOT raise

raised = False
try:
    die_if_unbearable('not-a-list', list[int])
except BeartypeDoorHintViolation:
    raised = True
assert raised
print('door API OK')

In [ ]:
# Example 5: BeartypeConf strategy configuration (decorator with custom conf).
from beartype import beartype, BeartypeConf, BeartypeStrategy
from beartype.roar import BeartypeCallHintParamViolation

# O1 strategy: constant-time checks (default).
conf_o1 = BeartypeConf(strategy=BeartypeStrategy.O1)

@beartype(conf=conf_o1)
def double(xs: list[int]) -> list[int]:
    return [x * 2 for x in xs]

assert double([1, 2, 3]) == [2, 4, 6]

raised = False
try:
    double('abc')  # type: ignore[arg-type]
except BeartypeCallHintParamViolation:
    raised = True

assert raised
assert isinstance(conf_o1, BeartypeConf)
print('BeartypeConf strategy OK')